[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C71_Prompt_Programming_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与一个可控的 in-context learning 模拟器

这个 notebook 做两件事：

1. **建起一个机制清楚的 ToyLM** —— demo 上的 kNN + 近因偏置 + 先验，三十行、确定性。
2. **在同一个任务上依次注入本课四个靶子**并量出代价：
   指令缺标签集（01）· 示例选错（02）· 顺序反了（02）· 没规定输出格式（01/04）。

> 心智模型：**一次调用里你在控制四个独立的东西——指令、示例、格式、解码约束。
> 它们的失败方式完全不同，所以需要四套不同的检查。**

## 0 · 环境

In [ ]:
import os, re, json, math, hashlib, itertools
from collections import Counter, defaultdict

import numpy as np

print('numpy', np.__version__)
print('本课全程 CPU / 断网 / 无需 API key —— 「模型」是一个确定性函数')

DIM = 2048

def tokenize(text):
    text = text.lower()
    toks = re.findall(r'[a-z0-9]+', text)
    for run in re.findall(r'[\u4e00-\u9fff]+', text):
        toks += list(run)
        toks += [run[i:i + 2] for i in range(len(run) - 1)]
    return toks

def embed(text, dim=DIM):
    v = np.zeros(dim)
    for tok in tokenize(text):
        v[int(hashlib.md5(tok.encode()).hexdigest(), 16) % dim] += 1.0
    n = np.linalg.norm(v)
    return v / n if n > 0 else v

def sim(a, b):
    return float(np.dot(embed(a), embed(b)))

## 1 · 任务与数据

一个五分类任务：把用户反馈分到 `bug / feature / billing / account / other`。
20 条训练样本（示例池）+ 10 条测试样本。

In [ ]:
LABELS = ['bug', 'feature', 'billing', 'account', 'other']

POOL = [
    ('登录后一直转圈，点不动', 'bug'),
    ('保存文件时报错 500', 'bug'),
    ('页面加载不出来', 'bug'),
    ('导出 CSV 会丢最后一行', 'bug'),
    ('希望支持批量导出', 'feature'),
    ('能不能加暗色主题', 'feature'),
    ('想要一个搜索框', 'feature'),
    ('建议增加导入模板', 'feature'),
    ('这个月扣了两次钱', 'billing'),
    ('发票开错了公司名', 'billing'),
    ('为什么涨价了', 'billing'),
    ('想申请退款', 'billing'),
    ('忘记密码收不到邮件', 'account'),
    ('想改绑定手机号', 'account'),
    ('账号被锁了', 'account'),
    ('想注销账号', 'account'),
    ('你们客服态度不错', 'other'),
    ('随便看看', 'other'),
    ('没什么事', 'other'),
    ('祝好', 'other'),
]

TEST = [
    ('打开报表就崩溃', 'bug'),
    ('保存草稿会丢内容', 'bug'),
    ('希望能加个批量删除', 'feature'),
    ('想要导出 PDF 的功能', 'feature'),
    ('这个月账单不对', 'billing'),
    ('发票上的税号不对', 'billing'),
    ('登录不上，密码重置也收不到', 'account'),
    ('手机号换了要怎么改', 'account'),
    ('感谢你们的帮助', 'other'),
    ('没别的了', 'other'),
]

print(f'示例池 {len(POOL)} 条（每类 {len(POOL) // len(LABELS)} 条）· 测试 {len(TEST)} 条')
print('标签分布:', Counter(y for _, y in POOL))

## 2 · ToyLM：demo 上的 kNN + 近因偏置 + 先验

三条机制，每一条都对应本课要研究的一件事：

| 机制 | 它解释了什么 |
|---|---|
| 在 demos 上做 kNN | **示例选择**为什么重要（它决定了近邻集合） |
| 近因偏置 `recency * i/(n-1)` | **顺序**为什么有影响 |
| 指令里没声明标签集 → `UNPARSEABLE` | **指令**与**格式**为什么是独立的失败 |

In [ ]:
UNPARSEABLE = 'UNPARSEABLE'

def toy_lm(instruction, demos, x, recency=0.35, prior='other'):
    """一个机制清楚的 in-context learning 替身。

    1) 指令里没声明标签集 → 模型「自创标签」，返回 UNPARSEABLE
    2) 没有 demos → 返回先验标签
    3) 否则在 demos 上做 kNN，分数叠加一个位置相关的近因偏置
    """
    declared = [l for l in LABELS if l in instruction]
    if not declared:
        return UNPARSEABLE
    if not demos:
        return prior
    E = np.stack([embed(t) for t, _ in demos])
    s = E @ embed(x)
    n = len(demos)
    boost = np.array([recency * (i / (n - 1) if n > 1 else 1.0) for i in range(n)])
    s = s + boost
    label = demos[int(np.argmax(s))][1]
    return label if label in declared else prior

INSTR_FULL = '把用户反馈分类为 bug / feature / billing / account / other 之一，只输出标签。'
INSTR_NO_LABELS = '请把用户反馈分类，只输出标签。'

def evaluate(instruction, demos, recency=0.35, test=None):
    test = TEST if test is None else test
    correct = unparsed = 0
    for x, gold in test:
        y = toy_lm(instruction, demos, x, recency=recency)
        unparsed += (y == UNPARSEABLE)
        correct += (y == gold)
    return dict(accuracy=correct / len(test), unparsed_rate=unparsed / len(test))

print('指令完整 + 全部示例:', evaluate(INSTR_FULL, POOL))
print('指令缺标签集         :', evaluate(INSTR_NO_LABELS, POOL))
assert evaluate(INSTR_FULL, POOL)['accuracy'] > 0.5
assert evaluate(INSTR_NO_LABELS, POOL)['unparsed_rate'] == 1.0
print('\n✅ 模拟器可用。注意第二行：**准确率是 0 而不是「差一点」**——')
print('   因为解析失败和答错是两个不同的失败，而它们经常被记成同一件事。')

## 3 · 一个最小的 prompt program

四个部件各自独立：`instruction` / `demos` / `format` / （`decoding` 在模块 04）。
注意 `parse` 是 program 的一部分——**格式规范与解析器必须成对定义**。

In [ ]:
class PromptProgram:
    """一个有签名的 prompt 模块。签名 = 输入字段 + 输出字段 + 输出取值域。"""

    def __init__(self, instruction, demos, out_field='label', allowed=None,
                 format_spec=None, recency=0.35):
        self.instruction = instruction
        self.demos = list(demos)
        self.out_field = out_field
        self.allowed = list(allowed) if allowed else list(LABELS)
        self.format_spec = format_spec          # None = 不规定格式
        self.recency = recency

    # ---- 渲染（真实系统里这里拼字符串） ----
    def render(self, x):
        parts = [self.instruction]
        if self.format_spec:
            parts.append(f'输出格式：{self.format_spec}')
        for dx, dy in self.demos:
            parts.append(f'输入：{dx}\n输出：{dy}')
        parts.append(f'输入：{x}\n输出：')
        return '\n\n'.join(parts)

    # ---- 调用 + 解析 ----
    def __call__(self, x):
        raw = toy_lm(self.instruction, self.demos, x, recency=self.recency)
        return raw, self.parse(raw)

    def parse(self, raw):
        """格式规范与解析器必须成对定义。没规定格式时，解析只能靠猜。"""
        if raw == UNPARSEABLE:
            return None
        if self.format_spec is None:
            # 没规定格式：只有恰好等于某个合法标签时才算解析成功
            return raw if raw in self.allowed else None
        if self.format_spec.startswith('json'):
            try:
                obj = json.loads(raw if raw.startswith('{')
                                 else json.dumps({self.out_field: raw}))
            except json.JSONDecodeError:
                return None
            v = obj.get(self.out_field)
            return v if v in self.allowed else None
        return raw if raw in self.allowed else None

    def signature(self):
        return dict(inputs=['x'], output=self.out_field, allowed=self.allowed,
                    n_demos=len(self.demos), has_format=self.format_spec is not None)

prog = PromptProgram(INSTR_FULL, POOL, format_spec='json {"label": "<标签>"}')
print('signature:', prog.signature())
print()
print('渲染出来的 prompt（前 160 字）:')
print(prog.render('打开报表就崩溃')[:160], '...')
print()
for probe in ['打开报表就崩溃', '这个月账单不对', '想改绑定手机号']:
    raw, parsed = prog(probe)
    print(f'{probe:<16} raw={raw!r:<12} parsed={parsed!r}')
    assert parsed in LABELS, '解析必须成功（对不对是另一件事）'
print('\n✅ 一个 prompt program = 指令 + 示例 + 格式 + 解析器 + 签名。')
print('   注意第一条预测是错的（应当是 bug）——**这正是本课要区分的两件事**：')
print('   解析成功（契约层面对了）与答对（能力层面对了）是两个独立的量。')
print('   签名是这四样东西的契约，模块 01 会展开它。')

## 4 · 靶子一（模块 01）：指令缺标签集

**格式合规与任务正确必须分开计。** 混在一起时，这个故障看起来像「效果差」。

In [ ]:
def scored(instruction, demos, **kw):
    """分别计三个量：解析成功率 / 解析成功里的正确率 / 端到端正确率。"""
    p = PromptProgram(instruction, demos, **kw)
    n_ok = n_correct_given_ok = n_correct = 0
    for x, gold in TEST:
        _, parsed = p(x)
        if parsed is not None:
            n_ok += 1
            n_correct_given_ok += (parsed == gold)
        n_correct += (parsed == gold)
    return dict(parse_rate=n_ok / len(TEST),
                acc_given_parsed=(n_correct_given_ok / n_ok) if n_ok else float('nan'),
                end_to_end=n_correct / len(TEST))

rows = {
    '指令完整':     scored(INSTR_FULL, POOL),
    '指令缺标签集': scored(INSTR_NO_LABELS, POOL),
}
print(f"{'配置':<14}{'解析成功率':>12}{'解析成功里的正确率':>20}{'端到端':>9}")
for k, v in rows.items():
    a = v['acc_given_parsed']
    print(f"{k:<14}{v['parse_rate']:>12.0%}"
          f"{('nan' if a != a else f'{a:.0%}'):>20}{v['end_to_end']:>9.0%}")

assert rows['指令完整']['parse_rate'] == 1.0
assert rows['指令缺标签集']['parse_rate'] == 0.0
print('\n✅ 两个配置的端到端分数都是一个数，但**它们的失败完全不同**：')
print('   指令缺标签集 → 解析成功率 0%，这不是能力问题，是契约问题。')
print('   如果只看端到端，你会去「优化 prompt 让模型更聪明」——而该做的是把标签集写进指令。')
print('   这就是 C03 模块 03 说的混淆变量：**答错，还是没按格式答？**')

## 5 · 靶子二（模块 02）：示例选错

同样的数量，不同的选择，准确率差一半。**而没有任何报错。**

In [ ]:
rng = np.random.default_rng(0)

def select_random(pool, k, seed=0):
    r = np.random.default_rng(seed)
    return [pool[i] for i in r.permutation(len(pool))[:k]]

def select_one_per_label(pool, k):
    out = []
    for l in LABELS:
        out += [t for t in pool if t[1] == l][:max(1, k // len(LABELS))]
    return out[:k]

def select_single_label(pool, k, label='other'):
    """最坏的选法：全部示例来自同一类。"""
    same = [t for t in pool if t[1] == label]
    rest = [t for t in pool if t[1] != label]
    return (same + rest)[:k]

print(f"{'示例选择（k=5）':<22}{'端到端正确率':>14}")
res = {}
for name, fn in [('随机 5 条', lambda: select_random(POOL, 5)),
                 ('每类 1 条', lambda: select_one_per_label(POOL, 5)),
                 ('全部来自 other 类', lambda: select_single_label(POOL, 5))]:
    d = select_random(POOL, 5) if False else fn()
    a = evaluate(INSTR_FULL, d)['accuracy']
    res[name] = a
    print(f'{name:<22}{a:>14.0%}')

assert res['每类 1 条'] > res['全部来自 other 类'], '覆盖全部标签的选法必须更好'
print(f"\n✅ 同样 5 条示例，覆盖全类 {res['每类 1 条']:.0%} vs 全来自一类 "
      f"{res['全部来自 other 类']:.0%}。")
print('   注意「全来自一类」这个配置在真实系统里不是刻意的——')
print('   它是「从最近的标注里取前 5 条」的自然结果，而最近的标注往往集中在同一批问题上。')

## 6 · 靶子三（模块 02）：顺序

同一批示例，只改顺序。近因偏置让**最后几条示例的标签被系统性偏好**。

In [ ]:
other_last = [t for t in POOL if t[1] != 'other'] + [t for t in POOL if t[1] == 'other']
other_first = [t for t in POOL if t[1] == 'other'] + [t for t in POOL if t[1] != 'other']

print(f"{'顺序':<16}{'端到端':>9}{'预测里 other 的占比':>22}")
def pred_dist(demos):
    ys = [toy_lm(INSTR_FULL, demos, x) for x, _ in TEST]
    return Counter(ys)

for name, d in [('other 放最后', other_last), ('other 放最前', other_first)]:
    a = evaluate(INSTR_FULL, d)['accuracy']
    c = pred_dist(d)
    print(f"{name:<16}{a:>9.0%}{c['other'] / len(TEST):>22.0%}")

a_last = evaluate(INSTR_FULL, other_last)['accuracy']
a_first = evaluate(INSTR_FULL, other_first)['accuracy']
c_last = pred_dist(other_last)['other'] / len(TEST)
c_first = pred_dist(other_first)['other'] / len(TEST)
assert a_last != a_first, '顺序会改变结果'
assert c_last > c_first, '靠后的示例的标签被系统性偏好'
print(f'\n✅ 只改顺序，端到端从 {a_first:.0%} 变成 {a_last:.0%}，')
print(f'   而 other 这个标签的预测占比从 {c_first:.0%} 变成 {c_last:.0%}。')
print('   **这不是随机波动，是一个有方向的偏置**——它可以被测量，也可以被利用或抵消（模块 02）。')
print('   工程含义：示例顺序必须固定并进指纹。「反正就是几个例子」是错的。')

## 7 · 靶子四（模块 01/04）：没规定输出格式

最麻烦的不是「全部解析失败」，而是**只有一部分失败**——
因为那会在你的评测里制造一个选择偏倚（模块 04 会展开）。

In [ ]:
# 模拟一个更真实的情形：模型有时输出裸标签，有时输出一句话
def toy_lm_verbose(instruction, demos, x, verbose_rate=0.4, seed=0, **kw):
    y = toy_lm(instruction, demos, x, **kw)
    if y == UNPARSEABLE:
        return y
    h = int(hashlib.md5((x + str(seed)).encode()).hexdigest(), 16)
    if (h % 100) / 100.0 < verbose_rate:
        return f'我认为这条反馈应该归类为 {y}。'      # 没规定格式时的自然输出
    return y

def scored_verbose(format_spec, verbose_rate=0.4):
    allowed = LABELS
    n_ok = n_correct = 0
    for x, gold in TEST:
        raw = toy_lm_verbose(INSTR_FULL, POOL, x, verbose_rate=verbose_rate)
        if format_spec is None:
            parsed = raw if raw in allowed else None          # 严格解析
        else:
            m = re.search('|'.join(allowed), raw)             # 规定了格式 → 抽取
            parsed = m.group(0) if m else None
        if parsed is not None:
            n_ok += 1
            n_correct += (parsed == gold)
    return dict(parse_rate=n_ok / len(TEST),
                end_to_end=n_correct / len(TEST))

strict = scored_verbose(None)
withfmt = scored_verbose('json {"label": "<标签>"}')
print(f"{'配置':<20}{'解析成功率':>12}{'端到端':>9}")
print(f'{"没规定格式":<20}{strict["parse_rate"]:>12.0%}{strict["end_to_end"]:>9.0%}')
print(f'{"规定格式 + 抽取":<20}{withfmt["parse_rate"]:>12.0%}{withfmt["end_to_end"]:>9.0%}')

assert 0.0 < strict['parse_rate'] < 1.0, '部分解析失败——这是最麻烦的情形'
assert withfmt['parse_rate'] > strict['parse_rate']
print(f"\n✅ 没规定格式时解析成功率 {strict['parse_rate']:.0%}——**部分失败而不是全部失败**。")
print('   为什么这更麻烦：如果你只在「解析成功」的样本上算准确率，')
print('   那么被丢掉的那些样本不是随机的（它们是模型更倾向于多说话的那些），')
print('   于是你的准确率估计是有偏的。模块 04 会把这个偏倚量出来。')

## 8 · 四个靶子的代价汇总

**注意最后一列**：四个靶子里只有一个能被「端到端准确率」这一个数正确诊断。

In [ ]:
SUMMARY = [
    ('01 指令缺标签集', '模型自创标签',       '解析成功率 0%',        '解析成功率'),
    ('02 示例选错',     '近邻集合覆盖不全',   '准确率 50%→20%，无报错', '按标签分层的准确率'),
    ('02 顺序反了',     '近因偏置',           '预测分布整体偏移',     '预测标签分布'),
    ('04 没规定格式',   '部分输出不可解析',   '**部分**解析失败',     '解析成功率 + 选择偏倚检查'),
]
print(f"{'靶子':<18}{'机制':<20}{'症状':<24}{'该看的信号'}")
for a, b, c, d in SUMMARY:
    print(f'{a:<18}{b:<20}{c:<24}{d}')

signals = {d for *_, d in SUMMARY}
assert len(signals) == 4, '四个靶子需要四个不同的信号'
print(f'\n✅ 四个靶子需要 {len(signals)} 个不同的信号——')
print('   端到端准确率会把它们全部压成一个数，而这个数不告诉你该改哪里。')
print('   这就是本课「把 prompt 拆成四个可分别检查的部件」的全部理由。')

## ✏️ 练习 1：分层的评分器

实现 `score(program, test)`，返回一个 dict：
- `parse_rate` —— 解析成功率
- `acc_given_parsed` —— 只在解析成功的样本上算的正确率（无样本时为 `float('nan')`）
- `end_to_end` —— 端到端正确率（解析失败记为错）
- `by_label` —— `{gold_label: 该类的端到端正确率}`
- `pred_dist` —— 预测标签的分布（`dict`，含 `None` 表示解析失败）

`by_label` 与 `pred_dist` 是诊断靶子二与靶子三的信号。

In [ ]:
def score(program, test=None):
    """返回 dict(parse_rate, acc_given_parsed, end_to_end, by_label, pred_dist)。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
good = PromptProgram(INSTR_FULL, POOL, format_spec='json {"label": "<标签>"}')
bad_instr = PromptProgram(INSTR_NO_LABELS, POOL, format_spec='json {"label": "<标签>"}')

s_good, s_bad = score(good), score(bad_instr)
print('好配置:', {k: v for k, v in s_good.items() if k not in ('by_label', 'pred_dist')})
print('  by_label:', {k: round(v, 2) for k, v in s_good['by_label'].items()})
print('  pred_dist:', s_good['pred_dist'])
print('坏指令:', {k: v for k, v in s_bad.items() if k not in ('by_label', 'pred_dist')})

assert s_good['parse_rate'] == 1.0
assert s_bad['parse_rate'] == 0.0
assert s_bad['acc_given_parsed'] != s_bad['acc_given_parsed'], '无样本时应当是 nan'
assert s_bad['end_to_end'] == 0.0
assert set(s_good['by_label']) == set(LABELS)
assert abs(np.mean(list(s_good['by_label'].values())) - s_good['end_to_end']) < 1e-9, \
    '每类样本数相同时，分层平均应当等于端到端'
assert sum(s_good['pred_dist'].values()) == len(TEST)
assert None in s_bad['pred_dist'] and s_bad['pred_dist'][None] == len(TEST)
print('✅ 练习 1 通过：五个量分别刻画四个靶子')

## 📖 参考答案 1

In [ ]:
# 练习 1 参考答案
def score(program, test=None):
    test = TEST if test is None else test
    n_ok = n_correct_given_ok = n_correct = 0
    by_label = defaultdict(list)
    preds = Counter()
    for x, gold in test:
        _, parsed = program(x)
        preds[parsed] += 1
        if parsed is not None:
            n_ok += 1
            n_correct_given_ok += (parsed == gold)
        hit = (parsed == gold)
        n_correct += hit
        by_label[gold].append(float(hit))
    return dict(
        parse_rate=n_ok / len(test),
        acc_given_parsed=(n_correct_given_ok / n_ok) if n_ok else float('nan'),
        end_to_end=n_correct / len(test),
        by_label={l: float(np.mean(v)) for l, v in by_label.items()},
        pred_dist=dict(preds))

s_good, s_bad = score(good), score(bad_instr)
assert s_good['parse_rate'] == 1.0 and s_bad['parse_rate'] == 0.0
assert s_bad['acc_given_parsed'] != s_bad['acc_given_parsed']
assert abs(np.mean(list(s_good['by_label'].values())) - s_good['end_to_end']) < 1e-9
assert s_bad['pred_dist'][None] == len(TEST)
print('✅ 参考答案 1 通过')
print('   注意 acc_given_parsed 在无样本时是 nan 而不是 0——')
print('   0 会被读成「解析成功的都答错了」，而真相是「没有解析成功的样本」。')
print('   这与 C68 模块 03 的「空集聚合返回 nan」是同一条纪律。')

## ✏️ 练习 2：program 指纹

按 C68 的规矩：**改了会让结果不可比较的东西，全部进指纹**。

prompt program 里这些是：指令、**示例的集合与顺序**、格式规范、
允许的取值域、近因参数、模型 ID、温度。

实现 `program_fingerprint(program, model_id, temperature)`，要求：
- 改指令 / 改格式 / 改取值域 / 改模型 / 改温度 → 指纹变
- **改示例顺序 → 指纹必须变**（这是最容易漏的一项）
- 与结果无关的东西（比如 program 上挂的日志字段）→ 指纹不变

In [ ]:
def program_fingerprint(program, model_id, temperature):
    """返回 12 位十六进制字符串。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
# ---- fixture 全部前置 ----
FMT = 'json {"label": "<标签>"}'
base = PromptProgram(INSTR_FULL, POOL[:6], format_spec=FMT)
reordered = PromptProgram(INSTR_FULL, list(reversed(POOL[:6])), format_spec=FMT)
subset = PromptProgram(INSTR_FULL, POOL[:5], format_spec=FMT)

# ---- 断言 ----
fp0 = program_fingerprint(base, 'toy-lm@1', 0.0)
print('基线指纹:', fp0)
assert program_fingerprint(reordered, 'toy-lm@1', 0.0) != fp0, '示例顺序必须进指纹'
assert program_fingerprint(subset, 'toy-lm@1', 0.0) != fp0, '示例集合必须进指纹'

for field, val in [('instruction', INSTR_NO_LABELS), ('format_spec', None),
                   ('allowed', ['bug', 'other']), ('recency', 0.0)]:
    p = PromptProgram(INSTR_FULL, POOL[:6], format_spec=FMT)
    setattr(p, field, val)
    assert program_fingerprint(p, 'toy-lm@1', 0.0) != fp0, f'{field} 必须进指纹'

assert program_fingerprint(base, 'toy-lm@2', 0.0) != fp0, '模型 ID 必须进指纹'
assert program_fingerprint(base, 'toy-lm@1', 0.7) != fp0, '温度必须进指纹'

# 与结果无关的字段
base.owner = 'team-nlp'; base.note = 'v3 手改'
assert program_fingerprint(base, 'toy-lm@1', 0.0) == fp0, '无关字段不该进指纹'
print('✅ 练习 2 通过：示例顺序也进了指纹')

## 📖 参考答案 2

In [ ]:
# 练习 2 参考答案
def program_fingerprint(program, model_id, temperature):
    payload = {
        'instruction': program.instruction,
        # 示例用**列表**而不是集合 —— 顺序必须影响指纹
        'demos': [[dx, dy] for dx, dy in program.demos],
        'format_spec': program.format_spec,
        'out_field': program.out_field,
        'allowed': list(program.allowed),
        'recency': program.recency,
        'model_id': model_id,
        'temperature': temperature,
    }
    blob = json.dumps(payload, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(blob.encode()).hexdigest()[:12]

fp0 = program_fingerprint(base, 'toy-lm@1', 0.0)
assert program_fingerprint(reordered, 'toy-lm@1', 0.0) != fp0
assert program_fingerprint(subset, 'toy-lm@1', 0.0) != fp0
assert program_fingerprint(base, 'toy-lm@2', 0.0) != fp0
assert program_fingerprint(base, 'toy-lm@1', 0.7) != fp0
print('✅ 参考答案 2 通过')
print('   唯一的技巧在 demos 那一行：**用列表而不是排序后的集合**。')
print('   第 6 节量过顺序有真实影响，所以「示例集合相同但顺序不同」')
print('   是两个不可比较的配置，指纹必须区分它们。')
print('   这与 C68 模块 02 的缓存键刚好相反——那里集合要排序，因为顺序不影响结果。')
print('   **判据始终是同一个：这个东西改了，结果会变吗？**')

## ✏️ 练习 3：逐项消融

实现 `ablate(base_program)`：逐一「关掉」四个部件，返回每一项的**边际贡献**
（关掉它之后端到端正确率的下降）。

关掉的定义：
- `instruction` → 换成 `INSTR_NO_LABELS`
- `demos` → 换成空列表
- `order` → 把示例顺序反转（不是关掉，是改成另一个顺序）
- `format` → `format_spec = None`

返回 `{部件名: 端到端下降的百分点}`，并额外给出 `base` 的端到端分数。

In [ ]:
def ablate(base_program):
    """返回 dict(base=..., instruction=..., demos=..., order=..., format=...)，
    后四项是「关掉这一项后端到端下降了多少」（可以为负）。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
BASE = PromptProgram(INSTR_FULL, other_last, format_spec='json {"label": "<标签>"}')
ab = ablate(BASE)
print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in ab.items()})

assert set(ab) == {'base', 'instruction', 'demos', 'order', 'format'}
assert ab['base'] > 0.5
# 关掉指令里的标签集 → 掉到 0，所以下降 = base
assert abs(ab['instruction'] - ab['base']) < 1e-9, '缺标签集时端到端是 0'
# 关掉示例 → 明显下降
assert ab['demos'] > 0.2
# 改顺序 → 有影响（可正可负，但不为 0）
assert abs(ab['order']) > 1e-9, '顺序必须有影响'
# 边际贡献排序：指令是最大的那一项
assert ab['instruction'] >= max(ab['demos'], abs(ab['order']), ab['format'])
print('✅ 练习 3 通过：四项的边际贡献都能被单独量出来')

## 📖 参考答案 3

In [ ]:
# 练习 3 参考答案
def ablate(base_program):
    def clone(**over):
        p = PromptProgram(base_program.instruction, base_program.demos,
                          out_field=base_program.out_field,
                          allowed=base_program.allowed,
                          format_spec=base_program.format_spec,
                          recency=base_program.recency)
        for k, v in over.items():
            setattr(p, k, v)
        return p

    base = score(base_program)['end_to_end']
    variants = {
        'instruction': clone(instruction=INSTR_NO_LABELS),
        'demos': clone(demos=[]),
        'order': clone(demos=list(reversed(base_program.demos))),
        'format': clone(format_spec=None),
    }
    out = {'base': base}
    for name, p in variants.items():
        out[name] = base - score(p)['end_to_end']
    return out

ab = ablate(BASE)
assert set(ab) == {'base', 'instruction', 'demos', 'order', 'format'}
assert abs(ab['instruction'] - ab['base']) < 1e-9
assert ab['demos'] > 0.2 and abs(ab['order']) > 1e-9
assert ab['instruction'] >= max(ab['demos'], abs(ab['order']), ab['format'])
print('✅ 参考答案 3 通过')
print('   消融是本课最常用的一个工具：它把「prompt 效果不好」变成一个有指向的结论。')
print('   两个读法上的注意：')
print('   ① order 那一项**可正可负**——「另一个顺序」不一定更差，')
print('      所以要报绝对值（它衡量的是敏感度，不是损失）；')
print('   ② 边际贡献不可加：四项之和通常不等于「全关掉」的损失（C69 模块 05 的二阶交互）。')

## ✏️ 练习 4：端到端归因器

给定一个表现不好的 program，判断**是哪一层的问题**。判定顺序（顺序本身就是结论）：

1. 解析成功率 < 1.0 → `'format'`（先修契约，不看效果）
2. 某个标签的 `by_label` 是 0，而示例里根本没有这个标签 → `'demos_coverage'`
3. 预测分布里某个标签的占比 > 0.5（而真实分布是均匀的）→ `'order_bias'`
4. 都不成立但端到端仍低于阈值 → `'capability'`
5. 端到端 ≥ 阈值 → `'ok'`

In [ ]:
def diagnose(program, threshold=0.6):
    """返回 'format' / 'demos_coverage' / 'order_bias' / 'capability' / 'ok'。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
# ---- fixture 全部前置 ----
FMT4 = 'json {"label": "<标签>"}'
p_fmt = PromptProgram(INSTR_NO_LABELS, POOL, format_spec=FMT4)
no_billing = [t for t in POOL if t[1] != 'billing']
p_cov = PromptProgram(INSTR_FULL, no_billing, format_spec=FMT4)
p_bias = PromptProgram(INSTR_FULL, select_single_label(POOL, 8, 'other'), format_spec=FMT4)
p_ok = PromptProgram(INSTR_FULL, POOL, format_spec=FMT4)

# ---- 断言 ----
assert diagnose(p_fmt) == 'format', diagnose(p_fmt)                  # 格式问题优先
assert diagnose(p_cov) == 'demos_coverage', diagnose(p_cov)          # 示例覆盖不全
d_bias = diagnose(p_bias)
assert d_bias in ('demos_coverage', 'order_bias'), d_bias            # 偏置
assert diagnose(p_ok, threshold=0.6) == 'ok', diagnose(p_ok, threshold=0.6)
assert diagnose(p_ok, threshold=0.95) == 'capability'                # 阈值抬高
print('✅ 练习 4 通过：归因器把失败指到具体的一层')

## 📖 参考答案 4

In [ ]:
# 练习 4 参考答案
def diagnose(program, threshold=0.6):
    s = score(program)
    if s['parse_rate'] < 1.0:
        return 'format'                                   # 先修契约
    demo_labels = {y for _, y in program.demos}
    for l, acc in s['by_label'].items():
        if acc == 0.0 and l not in demo_labels:
            return 'demos_coverage'
    n = sum(s['pred_dist'].values())
    for l, c in s['pred_dist'].items():
        if l is not None and c / n > 0.5:
            return 'order_bias'
    return 'ok' if s['end_to_end'] >= threshold else 'capability'

assert diagnose(p_fmt) == 'format'
assert diagnose(p_cov) == 'demos_coverage'
assert diagnose(p_bias) in ('demos_coverage', 'order_bias')
assert diagnose(p_ok, threshold=0.6) == 'ok'
assert diagnose(p_ok, threshold=0.95) == 'capability'
print('✅ 参考答案 4 通过')
print('   判定顺序 format → demos_coverage → order_bias → capability 就是本课模块的顺序，')
print('   也是真实调试时该走的顺序：**先修契约，再修示例，最后才怀疑「模型不够聪明」**。')
print('   反过来做（一上来就换更大的模型）是这一层最常见的浪费。')

## 🧪 真实工程胶囊：接真实模型

```python
# ══════════════════════════════════════════════════════════════════
# A. 签名与解析成对定义（讲解第 1 节 / 本 notebook 第 3 节）
# ══════════════════════════════════════════════════════════════════
from pydantic import BaseModel
from typing import Literal

class Classification(BaseModel):                  # ← 签名就是这个类
    label: Literal['bug', 'feature', 'billing', 'account', 'other']
    confidence: float

INSTRUCTION = (
    '把用户反馈分类。标签只能是 bug / feature / billing / account / other 之一。'
)   # ← 标签集必须在指令里出现（第 4 节：不写就是 100% 解析失败）

def render(x, demos, schema):
    parts = [INSTRUCTION, f'输出必须是 JSON，且符合此 schema：{schema}']
    for dx, dy in demos:                          # ← 顺序固定，进指纹
        parts.append(f'输入：{dx}\n输出：{dy.model_dump_json()}')
    parts.append(f'输入：{x}\n输出：')
    return '\n\n'.join(parts)

# ══════════════════════════════════════════════════════════════════
# B. 三个量分别记录（讲解第 2 节的「可测试」）
# ══════════════════════════════════════════════════════════════════
def run_one(x):
    raw = call_model(render(x, DEMOS, Classification.model_json_schema()))
    try:
        obj = Classification.model_validate_json(raw)
        return dict(raw=raw, parsed=obj, parse_ok=True)
    except Exception as e:
        return dict(raw=raw, parsed=None, parse_ok=False, err=str(e))
#   指标：parse_rate / acc_given_parsed / end_to_end —— **三个数一起报**。
#   只报最后一个时，「答错」与「没按格式答」被记成同一件事。

# ══════════════════════════════════════════════════════════════════
# C. 指纹（讲解第 3 节第 3 条 / 练习 2）
# ══════════════════════════════════════════════════════════════════
FP = sha256(json.dumps({
    'instruction': INSTRUCTION,
    'demos': [[dx, dy.model_dump()] for dx, dy in DEMOS],   # ← 列表，顺序敏感
    'schema': Classification.model_json_schema(),
    'model': 'claude-x@2026-06-01',                          # ← 钉死版本
    'temperature': 0.0,
    'decoding': {'constrained': True, 'grammar': 'json'},    # ← 模块 04
}, sort_keys=True)).hexdigest()[:12]
#   把 FP 写进每一条评测记录（C68-01）。**prompt 改一个字，FP 就必须变。**

# ══════════════════════════════════════════════════════════════════
# D. CI（讲解第 3 节）
# ══════════════════════════════════════════════════════════════════
# 阻断（确定性）：标签集出现在指令里；schema 与解析器一致；FP 变了但没记录变更
# 阻断（统计）  ：任一标签的分层准确率相对基线下降超过阈值（C68-04 推阈值）
# 报警          ：预测标签分布的 PSI（抓顺序偏置与示例漂移）
```

---

## 小结

| 结论 | 数字 / 判据 | 在哪一节 |
|---|---|---|
| 一次调用里在控制四个独立的东西 | 指令 / 示例 / 格式 / 解码约束 | 讲解 1 |
| 格式合规与任务正确必须分开计 | 缺标签集 → 解析率 0%，而不是「效果差」 | 第 4 节 |
| 示例选择的影响大于数量 | 同样 5 条：覆盖全类 50% vs 全来自一类 20% | 第 5 节 |
| 顺序是一个有方向的偏置，不是噪声 | 只改顺序，预测分布明显偏移 | 第 6 节 |
| 部分解析失败比全部失败更麻烦 | 它在评测里制造选择偏倚 | 第 7 节 |
| 四个靶子需要四个不同的信号 | 端到端准确率把它们压成一个数 | 第 8 节 |
| 示例的**顺序**必须进指纹 | 与缓存键的「集合要排序」刚好相反 | 练习 2 |
| 归因顺序：format → demos → order → capability | 先修契约，最后才怀疑模型 | 练习 4 |

下一模块：**01 · prompt program 与可组合结构**——
给每个模块一个签名，让「格式合规」与「答得对不对」变成两个可以独立测的量。